# Importing Libraries

In [1]:
## Import modules
import os, sys
import numpy as np
import geopandas as gpd
import cftime
import gc
import shapely
import json
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
import matplotlib.pyplot as plt

# Add the directory containing 'cmct' to the Python path
# Navigate two levels up to reach main CmCt dir
cmct_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))

# Initialising Logger
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Import utilities for this comparison
sys.path.insert(0, cmct_dir)
from cmct.time_utils import *
from cmct.calving import *
from cmct.calving_modules.interpolation import *
# from cmct.calving_modules.json_to_netcdf import *
# from cmct.shapefile_utils import *

# Force initial garbage collection
gc.collect()

40

In [2]:
# Reload modules to pick up any changes to imports
import importlib
import cmct.calving

importlib.reload(cmct.calving)

# Re-import to ensure functions are available
from cmct.calving import *


# CONFIGURATION

In [3]:
# Observation Dataset
# Ice sheet
loc = "GIS"  # 'GIS' or 'AIS'

# Set the observation data dir path
obs_filename = cmct_dir + "/data/calving/observed_icemask_ismip_annual.nc"

# To use aggregation functions for basin
basin_aggregation = True  # IMPORTANT

basin_filename = cmct_dir + "/bin/Calving/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp"

# Set the Model Data dir path
# model_filename = cmct_dir + "/test/calving/ensemble/sftgif_B001_hist.nc"
model_filename = cmct_dir + "/test/calving/sftgif_GIS_JPL_ISSM_historical.nc"

# Set time range for comparison
start_year = 2007
end_year = 2010

# List of basins (ex [NW, NE]) to compare if all -> "all", if none -> False
# If you do not know which basins are in the model, you can put "auto"
# NOTE: Align this list with the basins in the model.
basin_list = "all"

# Output filetype and filename
filetype = "netcdf"  # netcdf or json or None
filename = "calving_comparison"

# Optional Configurations
interpolation_method = "slinear"  # 'nearest', 'linear', 'cubic'
accuracy_calculation_method = "mean"  # 'mean', 'RMS',

colors = {
    "CW": "blue",
    "NE": "red",
    "SE": "green",
    "SW": "orange",
    "NO": "purple",
    "NW": "brown",
}


# Loading all data files

In [4]:
# Check if observation file exist
if not os.path.exists(obs_filename):
    raise FileNotFoundError(f"Observation file not found: {obs_filename}")

# # Check if model file exist
if not os.path.exists(model_filename):
    raise FileNotFoundError(f"Model file not found: {model_filename}")



if basin_aggregation and not os.path.exists(basin_filename):
    raise FileNotFoundError(f"Basin shapefile not found: {basin_filename}")
    # Load basin shapes

print(basin_filename)
basins = load_basins(basin_filename, basin_list)

print(obs_filename)
gsfc = load_gsfc_calving(obs_filename, basins)

print(model_filename)
model_res = load_model_calving(model_filename)

/Users/aditya_pachpande/Documents/GitHub/CmCt/bin/Calving/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp
/Users/aditya_pachpande/Documents/GitHub/CmCt/data/calving/observed_icemask_ismip_annual.nc
/Users/aditya_pachpande/Documents/GitHub/CmCt/test/calving/sftgif_GIS_JPL_ISSM_historical.nc


## Handelling Time Consistency

In [5]:
# Simplifying date data type
gsfc.ds["time"] = standardising_time_var(gsfc.time)
model_res.ds["time"] = standardising_time_var(model_res.time)

# Handelling Time Range
checking_calving_daterange(gsfc.time.values, model_res.time.values, start_year, end_year)


The selected dates 2007 to 2010 are within the overlapping data range.


# Interpolation

In [ ]:
interpolater = Interpolater(model_res, gsfc)
new_model_res = interpolater.interpolate()

del model_res 
gc.collect()

print(f"\nResampled data shape: {new_model_res.dims}")

2025-07-15 11:26:20,113 - INFO - Input x coordinates: [-720000. -715000. -710000. -705000. -700000. -695000. -690000. -685000.
 -680000. -675000. -670000. -665000. -660000. -655000. -650000. -645000.
 -640000. -635000. -630000. -625000. -620000. -615000. -610000. -605000.
 -600000. -595000. -590000. -585000. -580000. -575000. -570000. -565000.
 -560000. -555000. -550000. -545000. -540000. -535000. -530000. -525000.
 -520000. -515000. -510000. -505000. -500000. -495000. -490000. -485000.
 -480000. -475000. -470000. -465000. -460000. -455000. -450000. -445000.
 -440000. -435000. -430000. -425000. -420000. -415000. -410000. -405000.
 -400000. -395000. -390000. -385000. -380000. -375000. -370000. -365000.
 -360000. -355000. -350000. -345000. -340000. -335000. -330000. -325000.
 -320000. -315000. -310000. -305000. -300000. -295000. -290000. -285000.
 -280000. -275000. -270000. -265000. -260000. -255000. -250000. -245000.
 -240000. -235000. -230000. -225000. -220000. -215000. -210000. -20500


Resampled data shape: FrozenMappingWarningOnValuesAccess({'y': 2880, 'x': 1680, 'nv4': 4, 'time': 36})
Resampled data coordinates: ['lon', 'lat', 'time', 'x', 'y']


# Comparison and Residual Calculation 